# Olist E-Commerce — Exploratory Data Analysis

This notebook explores the raw Olist dataset to understand structure, quality, and key patterns before building the transformation pipeline.

**Sections:**
1. Dataset Overview
2. Distribution Analysis
3. Geographic Analysis
4. Time Series Analysis
5. Data Quality Report

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

RAW_DIR = Path("../data/raw")

---
## 1. Dataset Overview

The Olist dataset consists of 9 interrelated CSV files covering e-commerce transactions in Brazil from 2016 to 2018.

In [ ]:
files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translations": "product_category_name_translation.csv"
}

datasets = {name: pd.read_csv(RAW_DIR / f, low_memory=False) for name, f in files.items()}

print(f"{'Table':<15} {'Rows':>10} {'Columns':>8}  Dtypes")
print("-" * 70)
for name, df in datasets.items():
    dtypes = dict(df.dtypes.value_counts())
    print(f"{name:<15} {len(df):>10,} {len(df.columns):>8}  {dtypes}")

**Key observation:** The geolocation table has over 1 million rows — significantly larger than other tables. This will require deduplication during transformation. All date columns are stored as `object` (string) type and need parsing.

---
## 2. Distribution Analysis

### 2.1 Order Status Distribution

In [ ]:
orders = datasets["orders"]

fig, ax = plt.subplots(figsize=(10, 5))
status_counts = orders["order_status"].value_counts()
colors = ["#2ecc71" if s == "delivered" else "#95a5a6" for s in status_counts.index]
status_counts.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Order Status Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Count")
for i, v in enumerate(status_counts):
    ax.text(v + 500, i, f"{v:,}", va="center")
plt.tight_layout()
plt.show()

delivered_pct = 100 * status_counts.get("delivered", 0) / len(orders)
print(f"Delivered: {delivered_pct:.1f}% of all orders")

**Business insight:** The vast majority (~97%) of orders are delivered successfully. Canceled and unavailable orders are rare, suggesting a reliable fulfillment process.

### 2.2 Payment Method Distribution

In [ ]:
payments = datasets["payments"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pay_counts = payments["payment_type"].value_counts()
pay_counts.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Payment Count by Method", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=25)

pay_value = payments.groupby("payment_type")["payment_value"].sum().sort_values(ascending=False)
pay_value.plot(kind="bar", ax=axes[1], color="#e67e22")
axes[1].set_title("Revenue by Payment Method (R$)", fontweight="bold")
axes[1].set_ylabel("Total Value (R$)")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

**Business insight:** Credit card dominates both in transaction count and revenue. Boleto (Brazilian bank slip) is the second most popular method. Vouchers and debit cards are marginal.

### 2.3 Review Score Distribution

In [ ]:
reviews = datasets["reviews"]

fig, ax = plt.subplots(figsize=(8, 5))
score_counts = reviews["review_score"].value_counts().sort_index()
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"]
score_counts.plot(kind="bar", ax=ax, color=colors)
ax.set_title("Customer Review Score Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Score")
ax.set_ylabel("Count")
for i, v in enumerate(score_counts):
    ax.text(i, v + 500, f"{v:,}", ha="center")
plt.tight_layout()
plt.show()

avg_score = reviews["review_score"].mean()
print(f"Average review score: {avg_score:.2f}")
print(f"5-star reviews: {100 * score_counts.get(5, 0) / len(reviews):.1f}%")

**Business insight:** Reviews are heavily skewed toward 5 stars (positive), but there's a notable cluster at 1 star. This bimodal pattern suggests customers either love the experience or have strong complaints — likely tied to delivery performance.

### 2.4 Top Product Categories by Revenue

In [ ]:
items = datasets["items"]
products = datasets["products"]
translations = datasets["translations"]

merged = (
    items
    .merge(products, on="product_id")
    .merge(translations, on="product_category_name", how="left")
)
top_cats = (
    merged.groupby("product_category_name_english")["price"]
    .agg(["sum", "count"])
    .nlargest(12, "sum")
)

fig, ax = plt.subplots(figsize=(12, 6))
top_cats["sum"].plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 12 Product Categories by Revenue", fontsize=14, fontweight="bold")
ax.set_xlabel("Revenue (R$)")
plt.tight_layout()
plt.show()

**Business insight:** Bed/bath/table, health/beauty, and computers/accessories are the top revenue drivers. The long tail shows significant category diversity.

---
## 3. Geographic Analysis

### 3.1 Orders and Revenue by State

In [ ]:
customers = datasets["customers"]

order_customers = orders.merge(customers, on="customer_id")
order_items_full = order_customers.merge(items, on="order_id")

state_metrics = order_items_full.groupby("customer_state").agg(
    orders=("order_id", "nunique"),
    revenue=("price", "sum")
).sort_values("revenue", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

state_metrics["orders"].head(15).plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Orders by State (Top 15)", fontweight="bold")
axes[0].set_ylabel("Order Count")

state_metrics["revenue"].head(15).plot(kind="bar", ax=axes[1], color="#e67e22")
axes[1].set_title("Revenue by State (Top 15)", fontweight="bold")
axes[1].set_ylabel("Revenue (R$)")

plt.tight_layout()
plt.show()

sp_share = 100 * state_metrics.loc["SP", "revenue"] / state_metrics["revenue"].sum()
print(f"São Paulo alone accounts for {sp_share:.1f}% of total revenue")

**Business insight:** São Paulo (SP) dominates with ~37% of all revenue, followed by Rio de Janeiro (RJ) and Minas Gerais (MG). The Southeast region accounts for over 60% of the market. This concentration has implications for logistics optimization.

### 3.2 Seller Concentration

In [ ]:
sellers = datasets["sellers"]

fig, ax = plt.subplots(figsize=(12, 5))
seller_states = sellers["seller_state"].value_counts().head(15)
seller_states.plot(kind="bar", ax=ax, color="#27ae60")
ax.set_title("Sellers by State (Top 15)", fontsize=14, fontweight="bold")
ax.set_ylabel("Seller Count")
plt.tight_layout()
plt.show()

sp_sellers = 100 * seller_states.get("SP", 0) / len(sellers)
print(f"São Paulo sellers: {sp_sellers:.1f}% of total")

**Business insight:** Seller distribution mirrors customer distribution — SP dominates. This means most orders ship within the same state, which should correlate with faster delivery times.

---
## 4. Time Series Analysis

### 4.1 Monthly Order Volume (2016–2018)

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
monthly = orders.set_index("order_purchase_timestamp").resample("ME")["order_id"].count()

fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot(ax=ax, marker="o", color="steelblue", linewidth=2)
ax.fill_between(monthly.index, monthly.values, alpha=0.1, color="steelblue")
ax.set_title("Monthly Order Volume (2016–2018)", fontsize=14, fontweight="bold")
ax.set_ylabel("Orders")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

peak = monthly.idxmax().strftime("%B %Y")
print(f"Peak month: {peak} with {monthly.max():,} orders")

**Business insight:** Strong upward growth trend from late 2016 through 2018. November 2017 shows a spike likely driven by Black Friday promotions. The marketplace was clearly scaling rapidly during this period.

### 4.2 Delivery Time Distribution

In [ ]:
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)
delivered = orders.dropna(subset=["order_delivered_customer_date"]).copy()
delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"]
    - delivered["order_purchase_timestamp"]
).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delivered["delivery_days"].clip(0, 60).hist(
    bins=60, ax=axes[0], color="steelblue", edgecolor="white"
)
median_days = delivered["delivery_days"].median()
axes[0].axvline(
    median_days, color="red", linestyle="--",
    label=f"Median: {median_days:.0f} days",
)
axes[0].set_title("Delivery Time Distribution", fontweight="bold")
axes[0].set_xlabel("Days")
axes[0].set_ylabel("Orders")
axes[0].legend()

monthly_delivery = (
    delivered.set_index("order_purchase_timestamp")
    .resample("ME")["delivery_days"]
    .median()
)
monthly_delivery.plot(ax=axes[1], marker="o", color="#e67e22", linewidth=2)
axes[1].set_title("Median Delivery Time by Month", fontweight="bold")
axes[1].set_ylabel("Days")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

print(f"Median delivery: {delivered['delivery_days'].median():.0f} days")
print(f"95th percentile: {delivered['delivery_days'].quantile(0.95):.0f} days")

**Business insight:** Median delivery is about 12 days with a long tail extending past 30 days. Monthly median delivery time shows some seasonal variation, suggesting logistics capacity constraints during peak periods.

---
## 5. Data Quality Report

### 5.1 Null Analysis

In [ ]:
print(f"{'Table':<15} {'Column':<35} {'Nulls':>8} {'% Null':>8}")
print("-" * 70)
for name, df in datasets.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    for col, count in nulls.items():
        pct = 100 * count / len(df)
        print(f"{name:<15} {col:<35} {count:>8,} {pct:>7.1f}%")

**Data quality findings:**
- **Delivery dates** have nulls for undelivered orders — expected and handled by marking as NULL in the fact table
- **Review comments** are often null — filled with empty string during transformation
- **Product dimensions** have some nulls — kept as NULL (not critical for analytics)
- **order_approved_at** has 160 nulls — orders that weren't approved yet

### 5.2 Duplicate Analysis

In [ ]:
print(f"{'Table':<15} {'Total Rows':>12} {'Exact Dupes':>12} {'% Dupes':>8}")
print("-" * 50)
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    pct = 100 * dupes / len(df) if len(df) > 0 else 0
    flag = " ⚠️" if dupes > 0 else ""
    print(f"{name:<15} {len(df):>12,} {dupes:>12,} {pct:>7.1f}%{flag}")

**Data quality findings:**
- **Geolocation** is the biggest offender — massive duplication per zip code prefix (multiple lat/lng entries per zip). Resolved by keeping one entry per zip code during transformation
- Other tables have minimal or no exact duplicates

### 5.3 Type Inconsistencies

In [ ]:
date_columns = [
    ("orders", ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"]),
    ("reviews", ["review_creation_date", "review_answer_timestamp"]),
]

print("Columns stored as string that should be datetime:")
print("-" * 50)
for table_name, cols in date_columns:
    df = datasets[table_name]
    for col in cols:
        dtype = df[col].dtype
        print(f"  {table_name}.{col}: {dtype}")

print(f"\nAll {sum(len(c) for _, c in date_columns)} date columns are stored as 'object' (string)")
print("→ Resolved during transformation via pd.to_datetime()")

---
## Summary

**Key findings from this EDA that inform the pipeline design:**

| Finding | Impact on Pipeline |
|---|---|
| 9 related tables with clear join keys | Star schema with 5 dimensions + 1 fact table |
| 7 date columns stored as strings | `pd.to_datetime()` in transformation layer |
| 1M+ duplicate geolocations | Dedup to 1 row per zip code |
| Null delivery dates for undelivered orders | Keep as NULL, compute `delivery_days` only when available |
| Null review comments | Fill with empty string |
| Product categories in Portuguese | Merge with translation table |
| Credit card dominance in payments | Relevant for payment method analytics |
| Strong geographic concentration (SP ~37%) | Location dimension enables geographic queries |
| Growth trend 2016-2018 | Date dimension enables time series analytics |
| Bimodal review scores (1 and 5) | Delivery time correlation worth investigating |